<a href="https://colab.research.google.com/github/anshitatomar99-svg/Smart-Parking-Vision-YOLOv8/blob/main/smart_parking_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install yt-dlp
!pip install yt-dlp

# 2. Delete the old single-car video
!rm -f parking_video.mp4

# 3. Download the new multi-car video
!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best" "https://youtu.be/iYfJlDCygW4?si=BdvT7TvwTlQq5kJU" -o "parking_video.mp4"

[youtube] Extracting URL: https://youtu.be/iYfJlDCygW4?si=BdvT7TvwTlQq5kJU
[youtube] iYfJlDCygW4: Downloading webpage
[youtube] iYfJlDCygW4: Downloading visionos player API JSON
[youtube] iYfJlDCygW4: Downloading m3u8 information
[info] iYfJlDCygW4: Downloading 1 format(s): 401+140
[download] Destination: parking_video.f401.mp4
[download] 100% of  148.25MiB in 00:00:21 at 6.89MiB/s
[download] Destination: parking_video.f140.m4a
[download] 100% of  732.54KiB in 00:00:00 at 2.35MiB/s
[Merger] Merging formats into "parking_video.mp4"
Deleting original file parking_video.f140.m4a (pass -k to keep)
Deleting original file parking_video.f401.mp4 (pass -k to keep)


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 10.6 MB/s eta 0:00:00


In [ ]:
import cv2
import json
from ultralytics import YOLO

# 1. Load the basic YOLOv8 model
model = YOLO('yolov8n.pt')

# 2. Grab the first frame of your video
cap = cv2.VideoCapture("parking_video.mp4")
ret, frame = cap.read()
cap.release()

parking_data = []

if ret:
    # 3. Ask YOLO to find everything in the first frame
    results = model(frame)

    # 4. Loop through the results and grab the coordinates of every 'car' (Class 2)
    for box in results[0].boxes:
        if int(box.cls[0]) == 2:  # In YOLO, 2 means 'car'
            # Get the top-left (x1, y1) and bottom-right (x2, y2) of the car
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            # Convert it into the 4 corners our parking system needs
            four_corners = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]
            parking_data.append({"points": four_corners})

    # 5. Save it to the JSON file
    with open("bounding_boxes.json", "w") as f:
        json.dump(parking_data, f)

    print(f"Success! The AI automatically mapped {len(parking_data)} parking spots.")
else:
    print("Error reading the video.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

0: 384x640 5 cars, 5 buss, 1 truck, 4 boats, 2 cell phones, 9.0ms
Speed: 90.1ms preprocess, 9.0ms inference, 49.8ms postprocess per image at shape (1, 3, 384, 640)
Success! The AI automatically mapped 5 parking spots.


In [ ]:
import cv2
import json
import numpy as np
from ultralytics import YOLO

# 1. Load the YOLO model and your saved JSON coordinates
model = YOLO('yolov8n.pt')
with open("bounding_boxes.json", "r") as f:
    parking_data = json.load(f)

# Format the coordinates for OpenCV math
spots = [np.array(spot['points'], dtype=np.int32).reshape((-1, 1, 2)) for spot in parking_data]

# 2. Setup video input and output
cap = cv2.VideoCapture("parking_video.mp4")
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("smart_parking_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

print("Processing video using custom OpenCV logic... this will definitely work!")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 3. Ask YOLO to find all the cars in the frame
    results = model(frame, verbose=False)

    # Find the exact center dot (X, Y) of every car
    car_centers = []
    for box in results[0].boxes:
        if int(box.cls[0]) == 2:  # '2' is the YOLO ID code for 'car'
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            car_centers.append((float(cx), float(cy)))

    # 4. Check if those center dots are parked inside our spots
    available_spots = 0
    for spot in spots:
        is_occupied = False
        for cx, cy in car_centers:
            # OpenCV math checks if the car's center dot is inside the polygon boundary
            if cv2.pointPolygonTest(spot, (cx, cy), False) >= 0:
                is_occupied = True
                break

        # Draw Red if occupied, Green if empty
        color = (0, 0, 255) if is_occupied else (0, 255, 0)
        if not is_occupied:
            available_spots += 1

        cv2.polylines(frame, [spot], isClosed=True, color=color, thickness=3)

    # 5. Draw the live counter in the top left corner
    cv2.putText(frame, f"Available Spots: {available_spots}/{len(spots)}",
                (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    video_writer.write(frame)

cap.release()
video_writer.release()
print("Done! Download 'smart_parking_output.mp4' from your Colab files.")

Processing video using custom OpenCV logic... this will definitely work!
Done! Download 'smart_parking_output.mp4' from your Colab files.
